In [ ]:
import pandas as pd

example_data=pd.read_csv('D:/Biyesheji1/wangyesuiji1000/United_States.csv',header=None,names=['词条名称',"修改时间",'修改人','修改后长度','修改内容'])
example_data

In [ ]:
example_data.info()

In [ ]:
example_data=example_data.drop_duplicates()
example_data

In [ ]:
import re

# 定义一个函数，用于判断字符串是否为 IP 地址
def is_ip_address(s):
    ipv4_regex = r'^((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.)\
    {3}(25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$'
    ipv6_regex = r'^([0-9a-fA-F]{1,4}:){7}([0-9a-fA-F]{1,4})$'
    return re.match(ipv4_regex, s) is None and re.match(ipv6_regex, s) is None

# 对 DataFrame 中的某一列应用函数进行判断
example_data=example_data[example_data['修改人'].apply(lambda x: is_ip_address(str(x)))]
example_data=example_data.reset_index(drop=True)
example_data

In [ ]:
def is_ip_address(s):
    ipv4_regex = r'^((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.)\
    {3}(25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$'
    ipv6_regex = r'^([0-9a-fA-F]{1,4}:){7}([0-9a-fA-F]{1,4})$'
    return re.match(ipv4_regex, s) is None and re.match(ipv6_regex, s) is None

In [ ]:
def str_with_comma_to_int(str_num):
    # 去除逗号
    num_without_comma = str_num.replace(',', '')
    # 转换为整数
    num = int(num_without_comma)
    return num

# 测试示例
string_with_comma = "1,000,000"
result = str_with_comma_to_int(string_with_comma)
print(result)  # 输出结果为 1000000

In [ ]:
def has_number(input_string):
    if 'empty' not in input_string:
        for char in input_string:
            if char.isdigit():
                return True
    else:
        return False

In [ ]:
    for index, row in example_data.iterrows():
        if has_number(row['修改后长度']):
            #print(row['修改后长度'])
            index=row['修改后长度'].find(' ')
            row['修改后长度'] = (row['修改后长度'])[:index]
            row['修改后长度'] = str_with_comma_to_int(row['修改后长度'])
        else:
            row['修改后长度'] = None

In [ ]:
example_data=example_data[example_data['修改后长度'].notnull()]  
example_data=example_data.reset_index(drop=True)
example_data

In [ ]:
temp_list1=[0]+list(example_data['修改后长度'])
temp_list2=list(example_data['修改后长度'])+[0]

# 对应位置的数值做差
diff_list = [abs(x - y) for x, y in zip(temp_list1,temp_list2)]
diff_list=diff_list[1:-1]
diff_list.append(None)
example_data.insert(3,'更改长度',diff_list)
example_data

In [ ]:
example_data=example_data[example_data['更改长度']!=0]
example_data=example_data.reset_index(drop=True)
example_data

In [ ]:
temp_list1=['0']+list(example_data['修改人'])
temp_list2=list(example_data['修改人'])+['0']
len(temp_list1)

In [ ]:
# 将两个列表合并为DataFrame
edge_data=pd.DataFrame({'Source':temp_list1[1:-1],'Target':temp_list2[1:-1]})
edge_data.insert(2,'Weight',list(example_data['更改长度'])[:-1])
edge_data.insert(3,'Time',list(example_data['修改时间'])[:-1])
edge_data

In [ ]:
edge_data['Weight'].value_counts()

In [ ]:
edge_data['Source']==edge_data['Target']

In [ ]:
# 反转DataFrame并循环遍历
temp_list=list(edge_data['Weight'])
for index, row in edge_data[::-1].iterrows():
    if row['Source']==row['Target']:
        temp_list[index-1]=temp_list[index-1]+temp_list[index]
        
temp_list    

In [ ]:
edge_data['Weight']=temp_list
edge_data=edge_data[edge_data['Source']!=edge_data['Target']]
edge_data=edge_data.reset_index(drop=True)
edge_data

In [ ]:
edge_data

In [ ]:
# 按起始点和终点分组，并对权重进行累加
edge_data = edge_data.groupby(['Source', 'Target'], as_index=False).agg({'Weight': 'sum', 'Time': 'max'})
# 输出处理后的DataFrame
edge_data

In [ ]:
from sklearn import preprocessing

min_max_normalizer=preprocessing.MinMaxScaler(feature_range=(0,1))
#feature_range设置最大最小变换值，默认（0,1）
scaled_data=min_max_normalizer.fit_transform(edge_data[['Weight']])
#将数据缩放(映射)到设置固定区间

In [ ]:
scaled_data

In [ ]:
edge_data['scaled_Weight']=scaled_data
edge_data

In [ ]:
edge_data['Weight'].value_counts()

In [ ]:
edge_data[['Source','Target','Weight','scaled_Weight']].to_csv('edge_data.csv',index=None,header=['Source','Target','Weight','scaled_Weight'])

In [ ]:
# -*- coding: utf-8 -*-
import networkx as nx
import matplotlib.pyplot as plt
 
#定义有向图
DG = nx.DiGraph() 
#添加节点(列表)
DG.add_nodes_from(example_data['修改人'].unique())
print (DG.nodes())

In [ ]:
for index,row in edge_data.iterrows():
    row=edge_data.loc[index]
    DG.add_edges_from([(row['Source'],row['Target'], {'weight':row['Weight']})])

In [ ]:
adj_matrix = nx.adjacency_matrix(DG)
linjie_data=pd.DataFrame(adj_matrix.toarray())
linjie_data.to_csv('adj_matrix.csv',index=None,header=None)

In [ ]:
#绘制图形 设置节点名显示\节点大小\节点颜色
nx.draw(DG,with_labels=False, node_size=10)
plt.show()

In [ ]:
list(DG.nodes())

### 计算用户编辑指数

In [ ]:
edge_data

In [ ]:
import re
from datetime import datetime
date_format="%H:%M, %d %B %Y"
for index,row in edge_data.iterrows():
    edge_data.loc[index,'Time']=datetime.strptime(row['Time'],date_format)
    edge_data.loc[index,'Time']=edge_data.loc[index,'Time'].strftime("%Y-%m-%d %H:%M")
edge_data

In [ ]:
edge_data['Time']=pd.to_datetime(edge_data['Time'],format='%Y-%m-%d %H:%M')
edge_data.head()

In [ ]:
edge_data.loc[0,'Time']-edge_data.loc[1,'Time']

In [ ]:
df=edge_data[['Source','Time']]
df

In [ ]:
temp_data=edge_data[['Source','Time']]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 将时间列转换为日期时间格式
temp_data['Time'] = pd.to_datetime(temp_data['Time'])

# 按月份计数
temp_data['Month'] = temp_data['Time'].dt.to_period('M')
temp_data['Month'] = temp_data['Month'].astype(str)  # 转换为字符串类型

# 按月份分组，计算每个月份对应的用户数量
grouped = temp_data.groupby('Month').size().reset_index(name='User Count')

# 绘制折线图
plt.figure(figsize=(10, 6))
plt.plot(grouped['Month'], grouped['User Count'])
plt.xlabel('Month')
plt.ylabel('User Count')
plt.title('User Count vs. Month')
plt.xticks([])  # 去除x轴刻度
plt.show()

In [ ]:
print(list(range(5,9)))

In [ ]:
# 将时间列转换为日期时间格式
df['Time'] = pd.to_datetime(df['Time'])

# 提取小时
df['Hour'] = df['Time'].dt.hour

df['Hour'] = (df['Hour'] - 8) % 24

# 按小时分组，计算每个小时对应的用户数量
grouped = df.groupby('Hour').size().reset_index(name='User Count')

# 绘制折线图
plt.figure(figsize=(10, 6))
plt.plot(grouped['Hour'], grouped['User Count'], marker='o')
plt.xlabel('Hour')
plt.ylabel('User Count')
plt.title('User Count vs. Hour of the Day')
plt.xticks(range(24))  # 设置x轴刻度为一天24个小时
plt.grid(True)

plt.savefig('示例数据时间图.png')
plt.show()

In [ ]:
user_data=pd.DataFrame()

# 计算用户数据

In [ ]:
user_data=pd.DataFrame(edge_data['Source'].value_counts())
user_data=user_data.reset_index()
user_data.columns=['username','frequency']
user_data

In [ ]:
user_data['frequency']=user_data['frequency']-user_data['frequency'].mean()

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
r2=StandardScaler().fit_transform(pd.DataFrame(user_data['frequency']))#标准化处理
pd.DataFrame(r2)

In [ ]:
user_data=pd.concat([user_data,pd.DataFrame(r2)], axis=1)
user_data.columns=['username','frequency','frequency_num']
user_data

### 用户知识贡献量

In [ ]:
total_lengths=edge_data.groupby('Source')['Weight'].sum()

# 将用户名和总修改长度合并成一个新的 DataFrame
temp_data=pd.DataFrame({'用户名': total_lengths.index, '总修改长度': total_lengths.values})
temp_data.columns=['username','sum_length']
temp_data

In [ ]:
temp_data['sum_length'].value_counts()

In [ ]:
if 0 in list(temp_data['sum_length']):
    print(True)

In [ ]:
temp_data['sum_length']=temp_data['sum_length']-temp_data['sum_length'].mean()

In [ ]:
user_data=pd.merge(user_data,temp_data,on='username',how='left')

### 计算节点入度

In [ ]:
in_degree=pd.DataFrame(DG.in_degree)
in_degree.columns=['username','in_degree']
in_degree

In [ ]:
user_data=pd.merge(user_data,in_degree,on='username',how='left')
user_data

### 计算节点出度 

In [ ]:
out_degree=pd.DataFrame(DG.out_degree)
out_degree.columns=['username','out_degree']
out_degree

In [ ]:
user_data=pd.merge(user_data,out_degree,on='username',how='left')

In [ ]:
user_data

### 计算节点介数中心性 

In [ ]:
betweenness_centrality = nx.betweenness_centrality(DG)

In [ ]:
betweenness_centrality

In [ ]:
temp_data=pd.DataFrame({'username':list(betweenness_centrality.keys()),'betweenness_centrality':list(betweenness_centrality.values())})
temp_data.columns=['username','betweenness_centrality']
temp_data

In [ ]:
user_data=pd.merge(user_data,temp_data,on='username',how='left')

In [ ]:
user_data

###  聚集系数

In [ ]:
clustering_coefficient = nx.clustering(DG)
clustering_coefficient

In [ ]:
temp_data=pd.DataFrame({'username':list(clustering_coefficient.keys()),'clustering_coefficient':list(clustering_coefficient.values())})
temp_data.columns=['username','clustering_coefficient']
temp_data

In [ ]:
user_data=pd.merge(user_data,temp_data,on='username',how='left')
user_data

### 特征向量中心性

In [ ]:
eigenvector_centrality = nx.eigenvector_centrality(DG)
eigenvector_centrality

In [ ]:
temp_data=pd.DataFrame({'username':list(eigenvector_centrality.keys()),'eigenvector_centrality':list(eigenvector_centrality.values())})
temp_data.columns=['username','eigenvector_centrality']
temp_data

In [ ]:
user_data=pd.merge(user_data,temp_data,on='username',how='left')
user_data

### 点度中心性

In [ ]:
degree_centrality = nx.degree_centrality(DG)
degree_centrality

In [ ]:
temp_data=pd.DataFrame({'username':list(degree_centrality.keys()),'degree_centrality':list(degree_centrality.values())})
temp_data.columns=['username','degree_centrality']
temp_data

In [ ]:
user_data=pd.merge(user_data,temp_data,on='username',how='left')
user_data

### 接近中心性 

In [ ]:
closeness = nx.closeness_centrality(DG)
closeness

In [ ]:
temp_data=pd.DataFrame({'username':list(closeness.keys()),'closeness':list(closeness.values())})
temp_data.columns=['username','closeness']
temp_data

In [ ]:
user_data=pd.merge(user_data,temp_data,on='username',how='left')
user_data

###  用户平均编辑时间间隔

In [ ]:
edge_data['Time'].max()

In [ ]:
temp_list=[]
for key,group in edge_data[['Source','Time']].groupby('Source'):
    print(key)
    print(list(pd.DataFrame(group).iloc[0]))
    temp_list.append(list(pd.DataFrame(group).iloc[0]))

In [ ]:
temp_data=pd.DataFrame(temp_list)
temp_data.columns=['Source','Time']
temp_data

In [ ]:
temp_data['Time gap']=temp_data['Time'].max()-temp_data['Time']
temp_data=temp_data.sort_values(by='Time gap')
temp_data

In [ ]:
# 定义函数将 pd.datetime 对象转换为总分钟数
def pd_datetime_to_total_minutes(pd_dt):
    total_minutes = pd_dt.days * 24 * 60 + pd_dt.seconds // 60
    return total_minutes

# 使用 apply() 函数将函数应用到 DataFrame 的某一列
temp_data['Time gap'] = temp_data['Time gap'].apply(lambda x: pd_datetime_to_total_minutes(pd.to_timedelta(x)))

In [ ]:
temp_data.columns=['username','Time','Time gap']

In [ ]:
temp_data['Time gap']=temp_data['Time gap']/len(DG.nodes())

In [ ]:
user_data=pd.merge(user_data,temp_data,on='username',how='left')
user_data

### 计算结构洞指标

In [ ]:
A=nx.adjacency_matrix(DG).todense()

In [ ]:
import numpy as np
import networkx as nx

A=nx.adjacency_matrix(DG).todense()
#得到邻接矩阵
A=np.array(nx.adjacency_matrix(DG,nodelist=list(user_data['username'])).todense())

#转化为p_ij矩阵。p_ij代表节点i花费在节点j上的精力。
A=A/(A.sum(axis=0).reshape(-1,1))

A

In [ ]:

C=[]#保存各个节点的约束系数
for i in range(A.shape[0]):
    #知道当前节点的邻居节点
    n_idx=np.where(A[i]>0)[0]
    c_i=0
    for j in n_idx:
        #节点i和节点j的共同邻居
        com_n_idx=np.where(np.logical_and(A[i]>0,A[j]>0))[0]
        tmp=sum([A[i][k]*A[k][j] for k in com_n_idx])+A[i][j]
        c_i+=tmp*tmp
    C.append(c_i)
C

In [ ]:
import numpy as np
import networkx as nx

A = np.array(nx.adjacency_matrix(DG, nodelist=list(user_data['username'])).todense())

# 检查每列的和是否为零，避免除以零错误
sums = A.sum(axis=0)
for i in range(len(sums)):
    if sums[i] == 0:
        # 如果某列的和为零，则将该列所有元素都设为零
        A[:, i] = 0
    else:
        # 否则进行归一化操作，指定保持维度的方式
        A[:, i] = A[:, i] / sums[i]

C = []  # 保存各个节点的约束系数
for i in range(A.shape[0]):
    # 知道当前节点的邻居节点
    n_idx = np.where(A[i] > 0)[0]
    c_i = 0
    for j in n_idx:
        # 节点i和节点j的共同邻居
        com_n_idx = np.where(np.logical_and(A[i] > 0, A[j] > 0))[0]
        tmp = sum([A[i][k] * A[k][j] for k in com_n_idx]) + A[i][j]
        c_i += tmp * tmp
    C.append(c_i)
    
C

In [ ]:
C

In [ ]:
user_data['construct_label']=C

In [ ]:
user_data

In [ ]:
edge_data

### 判断节点社团演化属性

In [ ]:
edge_data[['Source','Target','scaled_Weight','Time']]

In [ ]:
import pandas as pd

# 定义时间段的分割函数
def segment_time(timestamps):
    # 找到时间的最小值和最大值
    min_time = min(timestamps)
    max_time = max(timestamps)
    
    # 计算时间段的长度（以秒为单位）
    time_range = (max_time - min_time).total_seconds()
    
    # 将时间段分成三个部分
    seg1_end = min_time + pd.Timedelta(seconds=(1/3) * time_range)
    seg2_end = min_time + pd.Timedelta(seconds=(2/3) * time_range)
    
    return seg1_end, seg2_end

# 将 datetime 列从字符串转换为 datetime64[ns] 格式
edge_data['Time'] = pd.to_datetime(edge_data['Time'])
df=edge_data[['Source','Target','scaled_Weight','Time']]

# 将时间戳转换为datetime对象
df['Time'] = pd.to_datetime(df['Time'])

# 提取时间戳并分段
timestamps = df['Time']
seg1_end, seg2_end = segment_time(timestamps)

print("第一段时间的结束时间：", seg1_end)
print("第二段时间的结束时间：", seg2_end)

In [ ]:
part1 = df[df['Time']<=seg1_end]
part2 = df[(df['Time']<=seg2_end)&(df['Time']>seg1_end)]
part3 = df[(df['Time']>seg2_end)]

part1=part1.reset_index(drop=True)
part2=part2.reset_index(drop=True)
part3=part3.reset_index(drop=True)

In [ ]:
part1

In [ ]:
list(part1['Source'].drop_duplicates())

In [ ]:
import community
def dynamic_process(part1,part2,part3):
    #定义有向图
    DG = nx.DiGraph() 
    #添加节点(列表)
    DG.add_nodes_from(list(part1['Source'].drop_duplicates()))
    
    for index,row in part1.iterrows():
        row=part1.loc[index]
        DG.add_edges_from([(row['Source'],row['Target'], {'weight':row['scaled_Weight']})])
    
    initial_partition = community.best_partition(DG.to_undirected(), weight='weight')
    
    #添加节点(列表)
    DG.add_nodes_from(list(part2['Source'].drop_duplicates()))
    
    for index,row in part2.iterrows():
        row=part2.loc[index]
        DG.add_edges_from([(row['Source'],row['Target'], {'weight':row['scaled_Weight']})])
    
    intermediate_partition = community.best_partition(DG.to_undirected(), weight='weight')
    
    #添加节点(列表)
    DG.add_nodes_from(list(part3['Source'].drop_duplicates()))
    
    for index,row in part3.iterrows():
        row=part3.loc[index]
        DG.add_edges_from([(row['Source'],row['Target'], {'weight':row['scaled_Weight']})])
    
    final_partition = community.best_partition(DG.to_undirected(), weight='weight')
    
    return(initial_partition,intermediate_partition,final_partition)

In [ ]:
initial_partition,intermediate_partition,final_partition=dynamic_process(part1,part2,part3)

In [ ]:
df

In [ ]:
all_nodes=list(df['Source'].drop_duplicates())
temp_list=[]
# 输出所有节点属于的社团类型以及趋势
for node in all_nodes:
    # 获取节点在第一个阶段的社团
    initial_community = initial_partition.get(node, None)
        
    # 获取节点在最后一个阶段的社团
    final_community = final_partition.get(node, None)
    # 获取节点在中间阶段的社团
    intermediate_community = intermediate_partition.get(node, None)

    if initial_community is not None:
        # 如果节点在第一个阶段有社团
        if final_community is not None:
            # 如果节点在最后一个阶段也有社团，根据社团的变化判断趋势
            if initial_community == final_community:
                trend = "稳定"
            elif list(initial_partition.values()).count(initial_community) < list(final_partition.values()).count(final_community):
                trend = "增长"
            elif list(initial_partition.values()).count(initial_community) > list(final_partition.values()).count(final_community):
                trend = "衰退"
        else:
            if intermediate_community is not None:
                # 如果节点在最后一个阶段也有社团，根据社团的变化判断趋势
                if initial_community == intermediate_community:
                    trend = "稳定"
                elif list(initial_partition.values()).count(initial_community) < list(intermediate_partition.values()).count(intermediate_community):
                    trend = "增长"
                elif list(initial_partition.values()).count(initial_community) > list(intermediate_partition.values()).count(intermediate_community):
                    trend = "衰退"
            else:
                trend = "衰退"
    else:
        if intermediate_community is not None:
            if final_community is not None:
                # 如果节点在最后一个阶段也有社团，根据社团的变化判断趋势
                if intermediate_community == final_community:
                    trend = "稳定"
                elif list(intermediate_partition.values()).count(intermediate_community) < list(final_partition.values()).count(final_community):
                    trend = "增长"
                elif list(intermediate_partition.values()).count(intermediate_community) > list(final_partition.values()).count(final_community):
                    trend = "衰退"
            else:
                trend = "衰退"
        else:#中间阶段社团为空
            trend = "稳定"
    
    # 输出节点属于的社团类型以及趋势
    print(f"节点{node}趋势为 {trend}")
    temp_list.append([node,trend])

In [ ]:
list(initial_partition.values()).count(initial_community)

In [ ]:
list(final_partition.values()).count(final_community)

In [ ]:
temp_data=pd.DataFrame(temp_list)
temp_data.columns=['username','dynamic_data']
temp_data

In [ ]:
user_data=pd.merge(user_data,temp_data,on='username',how='left')
user_data

In [ ]:
user_data['html']=example_data['词条名称']

In [ ]:
user_data

### 计算节点净入度

In [ ]:
user_data['pur_in_degree']=user_data['in_degree']-user_data['out_degree']

In [ ]:
len(DG.edges())

In [ ]:
user_data['pur_in_degree']=user_data['pur_in_degree']/len(DG.edges())
user_data.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 初始化MinMaxScaler
scaler = MinMaxScaler()

# 将数据转换为numpy数组并进行拟合
user_data['pur_in_degree']= scaler.fit_transform(user_data[['pur_in_degree']])
user_data.head()

### 使用louvain算法，进行社群划分，衡量用户知识异质性

In [ ]:
import networkx as nx
import community  # Louvain Algorithm 的实现
from collections import defaultdict


# 将有向图转换为无向图
G = DG.to_undirected()

# 使用 Louvain Algorithm 进行社群划分
partition = community.best_partition(G, weight='weight')

# 整理社群
communities = defaultdict(list)
for node, com in partition.items():
    communities[com].append(node)

# 输出社群
node_list=[]
for com, nodes in communities.items():
    print("Community:", nodes)
    node_list.append(nodes)

In [ ]:
node_list

In [ ]:
print(len(node_list))

In [ ]:
pd.DataFrame(node_list)

In [ ]:
import pandas as pd

# 统计每个字符串在 DataFrame 中出现在几列
def count_occurrences(df, search_list):
    occurrences = {}
    for item in search_list:
        count = 0
        for col in df.columns:
            if item in df[col].values:
                count += 1
        occurrences[item] = count
    return occurrences

# 执行统计并输出结果
result = count_occurrences(pd.DataFrame(node_list).T,list(user_data['username']))
print(result)

In [ ]:
temp_data=pd.DataFrame({'username':result.keys(), 'know_diff':result.values()})
temp_data

In [ ]:
temp_data['know_diff'].value_counts()